In [ ]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
# Change this for each of your 4 notebooks
MODEL_TYPE = "TST"  # Options: "MetaCNNLSTM", "DeepCNNLSTM", "TST", "ContrastiveNet"

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_mamlpp100_{MODEL_TYPE}_2fcv_hpo"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_mamlpp100_TST_2fcv_hpo
From path: C:\Users\kdmen\Repos\pers-gest-cls\optuna_journals\1s3w_mamlpp100_TST_2fcv_hpo.log


In [ ]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'mamlpp_pretr_TST_2fcv_hpo'


In [11]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 103 trials.
Best value (Accuracy): 0.7281
Best Hyperparameters:
  meta_batchsize: 32
  maml_inner_steps: 5
  maml_inner_steps_eval: 30
  maml_alpha_init: 0.0020499001022000187
  maml_alpha_init_eval: 0.008272056156701605
  outer_lr: 0.0003058672277984587
  wd: 9.876855843853168e-05
  groupnorm_num_groups: 8
  use_GlobalAvgPooling: True
  finetuning_approach: full
  best_or_last_pretr: last
  context_emb_dim: 8
  context_pool_type: mean
  episodes_per_epoch_train: 500
  optimizer: adam
  use_maml_msl: hybrid
  maml_msl_num_epochs: 40


In [12]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [13]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [14]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [15]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [27]:
# FULL
params_to_plot = ["best_or_last_pretr", "context_emb_dim", "context_pool_type", "episodes_per_epoch_train", "finetuning_approach", "groupnorm_num_groups", 
                  "maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "meta_batchsize", "optimizer", "use_GlobalAvgPooling", "use_maml_msl", "wd"]

# PRETRAINING STUFF / MISC
# Apparently finetuning_approach and use_maml_msl only had 1 value? Yes for finetuning currently, but use_maml_msl should have had true and false as well no? ...
params_to_plot_MISC = ["best_or_last_pretr", "finetuning_approach", "use_maml_msl"]

# ARCHITECTURE? --> What does this even mean? How can this vary if we are using the pretrained model....
params_to_plot_ARCH = ["context_emb_dim", "context_pool_type", "groupnorm_num_groups", "use_GlobalAvgPooling"]

# MAML++
params_to_plot_MAMLPP = ["maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "use_maml_msl"]

# LRs MAML++
params_to_plot_LRS = ["maml_alpha_init", "maml_alpha_init_eval", "outer_lr"]

# Learning HPs
params_to_plot_HPS = ["episodes_per_epoch_train", "meta_batchsize", "optimizer", "wd"]

In [28]:
from optuna.visualization import plot_slice


In [29]:
fig_slice = plot_slice(study, params=params_to_plot_MISC)
fig_slice.show()

In [30]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [31]:
fig_slice = plot_slice(study, params=params_to_plot_MAMLPP)
fig_slice.show()

In [32]:
fig_slice = plot_slice(study, params=params_to_plot_LRS)
fig_slice.show()

In [33]:
fig_slice = plot_slice(study, params=params_to_plot_HPS)
fig_slice.show()

In [16]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
82,82,0.728125,0.725000,2026-03-26 06:07:44.665186,0 days 00:35:19.250255
85,85,0.703646,0.707812,2026-03-26 06:43:40.385679,0 days 00:33:56.694325
83,83,0.699479,0.701042,2026-03-26 06:15:50.786493,0 days 00:28:29.050016
86,86,0.696875,0.706771,2026-03-26 06:44:41.322725,0 days 01:04:03.307863
99,99,0.695833,0.705729,2026-03-26 09:24:57.414620,0 days 00:25:31.666735
40,40,0.680729,0.681771,2026-03-26 00:24:45.184811,0 days 00:29:42.147793
87,87,0.678125,0.696354,2026-03-26 06:45:42.034133,0 days 01:10:24.541326
100,100,0.677604,0.683333,2026-03-26 09:29:33.567015,0 days 00:15:37.286139
73,73,0.675521,0.689063,2026-03-26 04:26:32.402323,0 days 01:05:30.500876
79,79,0.671354,0.677604,2026-03-26 05:24:44.316848,0 days 00:51:55.216477
